# Image Forgery Detection & Localization — End-to-End (Đề tài 2)

Một notebook chạy **toàn bộ** quy trình, không cần notebook cũ:

`môi trường → CASIA v2 + manifest → train U-Net–ResNet34 (RGB / Noise / DCT) + SegFormer → chọn ngưỡng trên validation → test CASIA → test Coverage (cross-dataset) → bảng kết quả → dự đoán + web demo`

**Cách dùng:** `Runtime → Change runtime type → GPU`, sửa cell **CONFIG**, rồi `Runtime → Run all`.

- Mọi checkpoint/kết quả lưu trên Google Drive. Bị ngắt thì *Run all* lại: lượt nào xong thì bỏ qua, lượt dở thì resume từ `last.pt`.
- Checkpoint U-Net cũ trong `checkpoints/controlled_multiseed_v1` được **dùng lại** nếu đã đủ 10 epoch.
- Test set chỉ được dùng sau khi checkpoint và ngưỡng đã chốt trên validation.

## 0. CONFIG — chỉ cần sửa cell này

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ImageForgeryResearch')
REPO_URL   = 'https://github.com/ngocan-dev/image-forgery-localization.git'
REPO       = Path('/content/image-forgery-localization')

# Dữ liệu CASIA v2
CASIA_ZIP        = DRIVE_ROOT / 'datasets/CASIA2.zip'          # dự phòng khi không có KaggleHub
KAGGLE_DATASET   = 'chongtrung/casia-v2'
LOCAL_CASIA      = Path('/content/datasets/CASIA2')
MANIFEST_BACKUP  = DRIVE_ROOT / 'manifests/baseline_v3'        # train.csv, val.csv, test.csv
MANIFEST_PREFIX  = '/kaggle/input/casia-v2/CASIA2/'            # tiền tố đường dẫn trong manifest gốc
LOCAL_MANIFEST   = Path('/content/manifests/casia2')

# Thí nghiệm
SEEDS            = [7, 21, 42]
UNET_MODES       = ['rgb', 'noise', 'frequency']   # RGB vs RGB+Noise vs RGB+DCT
SEGFORMER_ARCH   = 'segformer_mit_b2'               # Transformer upgrade ('segformer_mit_b0' nếu GPU yếu)
SEGFORMER_MODES  = ['rgb']                          # thêm 'noise'/'frequency' nếu còn thời gian
# 10 = dùng lại 9 checkpoint U-Net đã có (đã đủ 10 epoch), chỉ train thêm SegFormer.
# Lớn hơn (vd. 30) = nghiên cứu train lâu hơn: train lại TẤT CẢ cấu hình với cùng số epoch, thư mục riêng.
EPOCHS           = 10

# Checkpoint & kết quả (tách theo số epoch để không trộn hai protocol)
LEGACY_UNET_ROOT = DRIVE_ROOT / 'checkpoints/controlled_multiseed_v1'   # {mode}_seed{seed}/ (9 lượt đã có)
LEGACY_MANIFEST  = DRIVE_ROOT / 'final_results/casia_stage1_3modes_3seeds_v1/final_manifest.json'  # sha256 + kết quả test cũ
E2E_ROOT         = DRIVE_ROOT / f'checkpoints/topic2_e2e_{EPOCHS}ep'    # các lượt train mới
EVAL_DIR         = DRIVE_ROOT / f'reports/eval_{EPOCHS}ep'
TABLE_DIR        = DRIVE_ROOT / f'reports/tables_{EPOCHS}ep'

# Bật/tắt từng giai đoạn
RUN_TRAIN_UNET      = True
RUN_TRAIN_SEGFORMER = True
RUN_CASIA_TEST      = True
RUN_COVERAGE        = True
COVERAGE_MASKS_CHECKED = False   # đặt True SAU KHI đã xem ảnh preview mask Coverage ở mục 6
RUN_DEMO            = True
UPLOAD_OWN_IMAGE    = False  # True: mục 9 hiện nút upload ảnh (Run all sẽ dừng chờ chọn file)

## 1. GPU, Google Drive, mã nguồn

In [ ]:
import os, sys, subprocess, shutil, json, zipfile
from pathlib import PurePosixPath

def run(cmd, cwd=None):
    """Chạy lệnh, in log trực tiếp, dừng notebook nếu lỗi."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, cwd=cwd or REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError(f'Command failed ({process.returncode}): {" ".join(cmd)}')

import torch
assert torch.cuda.is_available(), 'Bật GPU: Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| Python', sys.version.split()[0])

from google.colab import drive
drive.mount('/content/drive')
for p in (E2E_ROOT, EVAL_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

if (REPO / '.git').is_dir():
    run(['git', 'pull', '--ff-only'])
else:
    run(['git', 'clone', REPO_URL, REPO], cwd='/content')
run(['git', 'log', '--oneline', '-1'])

In [ ]:
# Cài package (giữ torch GPU của Colab; project khai báo Python 3.11 nên bỏ qua kiểm tra phiên bản)
run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.6.0', 'gradio', 'pytest'])
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python', '-e', '.'])
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
import segmentation_models_pytorch as smp
print('smp', smp.__version__)
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider'])

## 2. CASIA v2 và manifest train / val / test

In [ ]:
def casia_ready(path):
    return all((path / d).is_dir() for d in ('Au', 'Tp', 'Groundtruth'))

LOCAL_CASIA.parent.mkdir(parents=True, exist_ok=True)
if not casia_ready(LOCAL_CASIA):
    if LOCAL_CASIA.is_symlink():
        LOCAL_CASIA.unlink()
    source = None
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        candidates = [downloaded] + [p for p in downloaded.rglob('*') if p.is_dir()]
        source = next((p for p in candidates if casia_ready(p)), None)
    except Exception as error:
        print('KaggleHub không dùng được:', error)
    if source is not None:
        LOCAL_CASIA.symlink_to(source, target_is_directory=True)
        print('Dùng KaggleHub:', source)
    else:
        assert CASIA_ZIP.is_file(), f'Không có KaggleHub và không thấy {CASIA_ZIP}'
        if LOCAL_CASIA.exists():
            shutil.rmtree(LOCAL_CASIA)
        with zipfile.ZipFile(CASIA_ZIP) as archive:
            for item in archive.infolist():   # chặn đường dẫn nguy hiểm trong ZIP
                parts = PurePosixPath(item.filename).parts
                assert parts and parts[0] == 'CASIA2' and '..' not in parts and not item.filename.startswith('/'), item.filename
            archive.extractall(LOCAL_CASIA.parent)
        print('Giải nén từ Drive xong')
assert casia_ready(LOCAL_CASIA)
print('CASIA2:', LOCAL_CASIA.resolve())

In [ ]:
import pandas as pd
LOCAL_MANIFEST.mkdir(parents=True, exist_ok=True)
new_prefix = str(LOCAL_CASIA) + '/'
for name in ('train', 'val', 'test'):
    frame = pd.read_csv(MANIFEST_BACKUP / f'{name}.csv', keep_default_na=False)
    for column in ('image_path', 'mask_path'):
        def convert(path):
            if not path:
                return ''
            assert path.startswith(MANIFEST_PREFIX), f'Đường dẫn lạ: {path}'
            return new_prefix + path[len(MANIFEST_PREFIX):]
        frame[column] = frame[column].map(convert)
    missing = [p for p in list(frame['image_path']) + [m for m in frame['mask_path'] if m] if not Path(p).is_file()]
    assert not missing, f'{name}: thiếu {len(missing)} file, ví dụ {missing[:2]}'
    frame.to_csv(LOCAL_MANIFEST / f'{name}.csv', index=False)
    print(f'{name.upper():5s} {len(frame):5d} ảnh | {frame["label"].value_counts().to_dict()}')
TRAIN, VAL, TEST = (LOCAL_MANIFEST / f'{s}.csv' for s in ('train', 'val', 'test'))

In [ ]:
# Kiểm tra: split hiện tại phải TRÙNG split đã dùng để train các checkpoint cũ, và 3 tập không chồng lấn.
import hashlib
from forgery_localization.datasets.casia_dataset import CASIADataset, validate_split_isolation

def legacy_split_signature(dataset):
    fields = ['image_hash', 'label', 'source_group', 'mask_verified', 'mask_source', 'mask_threshold', 'threshold_approved']
    rows = [{**{k: str(r.get(k, '')) for k in fields},
             'mask_name': Path(r['mask_path']).name if r['mask_path'] is not None else ''} for r in dataset.records]
    return hashlib.sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()

train_ds = CASIADataset(TRAIN, root='/', image_size=(256, 256))
val_ds = CASIADataset(VAL, root='/', image_size=(256, 256))
test_ds = CASIADataset(TEST, root='/', image_size=(256, 256))
validate_split_isolation(train_ds, val_ds, test_ds)
print('Không chồng lấn group / path / hash giữa train, val, test: PASS')

legacy_configs = sorted(LEGACY_UNET_ROOT.glob('*_seed*/config.json'))
if legacy_configs:
    reference = json.loads(legacy_configs[0].read_text())
    assert legacy_split_signature(train_ds) == reference['train_signature'], 'Train split KHÁC split của checkpoint cũ!'
    assert legacy_split_signature(val_ds) == reference['val_signature'], 'Val split KHÁC split của checkpoint cũ!'
    print('Split trùng với checkpoint cũ: PASS')
else:
    print('Chưa có checkpoint cũ — mọi lượt sẽ train mới.')
del train_ds, val_ds   # test_ds giữ lại cho phần demo

## 3. Train

* **U-Net–ResNet34** × {RGB, RGB+Noise, RGB+DCT} × 3 seed = 9 lượt. Lượt cũ đủ 10 epoch thì dùng lại, thiếu thì train bằng `scripts/train_multiinput.py` (cùng protocol: weighted BCE + Dice, flip xác định theo seed, chọn checkpoint theo val micro Dice@0.5).
* **SegFormer** (Transformer) cùng protocol để so sánh CNN và Transformer.

In [ ]:
def run_complete(directory, epochs=EPOCHS):
    history = directory / 'history.csv'
    # Đúng bằng số epoch của protocol (không dùng lẫn lượt 10 epoch cho nghiên cứu 30 epoch)
    return (directory / 'best_dice.pt').is_file() and history.is_file() and \
        int(pd.read_csv(history)['epoch'].max()) == epochs

def checkpoint_for(arch, mode, seed):
    """Đường dẫn best_dice.pt đã hoàn tất, ưu tiên lượt cũ của notebook; None nếu chưa có."""
    if arch == 'unet_resnet34':
        legacy = LEGACY_UNET_ROOT / f'{mode}_seed{seed}'
        if run_complete(legacy):
            return legacy / 'best_dice.pt'
    new = E2E_ROOT / f'{arch}_{mode}_seed{seed}'
    return new / 'best_dice.pt' if run_complete(new) else None

PLAN = [('unet_resnet34', m, s) for s in SEEDS for m in UNET_MODES] + \
       [(SEGFORMER_ARCH, m, s) for s in SEEDS for m in SEGFORMER_MODES]

def show_status():
    rows = [dict(architecture=a, mode=m, seed=s, checkpoint=str(checkpoint_for(a, m, s) or '— chưa có'))
            for a, m, s in PLAN]
    display(pd.DataFrame(rows))
show_status()

In [ ]:
# Kiểm tra toàn vẹn 9 checkpoint cũ: sha256 phải khớp final_manifest.json (bản đã chốt ngày 17/11)
import hashlib
LEGACY_RESULTS = {}
if LEGACY_MANIFEST.is_file():
    frozen = json.loads(LEGACY_MANIFEST.read_text())
    expected = {f['path']: f['sha256'] for f in frozen['files']}
    LEGACY_RESULTS = {f"{c['mode']}_seed{c['seed']}": c for c in frozen['checkpoints']}
    rows = []
    for key in LEGACY_RESULTS:
        path = LEGACY_UNET_ROOT / key / 'best_dice.pt'
        digest = hashlib.sha256()
        with open(path, 'rb') as stream:
            for block in iter(lambda: stream.read(1 << 22), b''):
                digest.update(block)
        rows.append(dict(run=key, sha256_ok=digest.hexdigest() == expected[f'checkpoints/{key}/best_dice.pt'],
                         selected_epoch=LEGACY_RESULTS[key]['selected_epoch']))
    integrity = pd.DataFrame(rows); display(integrity)
    assert integrity['sha256_ok'].all(), 'Checkpoint cũ đã bị thay đổi so với bản chốt!'
else:
    print('Không thấy', LEGACY_MANIFEST, '— bỏ qua kiểm tra sha256')

In [ ]:
def train(arch, mode, seed):
    if checkpoint_for(arch, mode, seed) is not None:
        print(f'SKIP {arch} {mode} seed {seed}: đã hoàn tất'); return
    run([sys.executable, 'scripts/train_multiinput.py', '--architecture', arch, '--mode', mode, '--seed', seed,
         '--train-manifest', TRAIN, '--val-manifest', VAL, '--root', '/', '--output-root', E2E_ROOT,
         '--epochs', EPOCHS])

if RUN_TRAIN_UNET:
    for seed in SEEDS:
        for mode in UNET_MODES:
            train('unet_resnet34', mode, seed)

In [ ]:
if RUN_TRAIN_SEGFORMER:
    for seed in SEEDS:
        for mode in SEGFORMER_MODES:
            train(SEGFORMER_ARCH, mode, seed)
show_status()

In [ ]:
# Đường cong học (validation) của mọi lượt đã xong
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for arch, mode, seed in PLAN:
    ckpt = checkpoint_for(arch, mode, seed)
    if ckpt is None:
        continue
    history = pd.read_csv(ckpt.parent / 'history.csv')
    label = f'{arch.split("_")[0]}-{mode}-s{seed}'
    axes[0].plot(history['epoch'], history['val_micro_dice'], label=label)
    axes[1].plot(history['epoch'], history['val_loss'], label=label)
axes[0].set_title('Validation micro Dice @0.5'); axes[1].set_title('Validation loss')
for ax in axes: ax.set_xlabel('epoch'); ax.grid(alpha=.3)
axes[0].legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## 4. Chọn ngưỡng Real/Fake trên VALIDATION

Ảnh là **Forged** nếu `image_score` (trung bình 1% pixel nghi ngờ nhất) ≥ ngưỡng. Ngưỡng tối đa F1 trên val, lưu `thresholds.json` cạnh checkpoint và **đóng băng** cho mọi tập test.

In [ ]:
CHECKPOINTS = [str(c) for c in (checkpoint_for(a, m, s) for a, m, s in PLAN) if c is not None]
print(len(CHECKPOINTS), 'checkpoint hoàn tất'); print(*CHECKPOINTS, sep='\n')
assert CHECKPOINTS, 'Chưa có checkpoint nào'

def evaluate(manifest, dataset_name, extra=()):
    run([sys.executable, 'scripts/evaluate.py', '--checkpoint', *CHECKPOINTS, '--manifest', manifest,
         '--root', '/', '--dataset-name', dataset_name, '--output-dir', EVAL_DIR, *extra])

evaluate(VAL, 'casia_val', ['--calibrate'])

## 5. Test CASIA v2 (lần đầu chạm vào test set)

In [ ]:
if RUN_CASIA_TEST:
    evaluate(TEST, 'casia_test', ['--isolation-manifests', TRAIN, VAL, '--save-visuals', '8'])

In [ ]:
# Đối chiếu: code mới (fp32) phải tái tạo micro Dice test của báo cáo cũ (fp16 autocast) — sai khác nhỏ là bình thường
rows = []
for key, old in LEGACY_RESULTS.items():
    summary_path = EVAL_DIR / 'casia_test' / f'unet_resnet34_{key}' / 'summary.json'
    if summary_path.is_file():
        new = json.loads(summary_path.read_text())['pixel']
        rows.append(dict(run=key, old_micro_dice=old['test_micro_dice'], new_micro_dice=new['micro_dice'],
                         diff=new['micro_dice'] - old['test_micro_dice'],
                         old_mean_tampered_dice=old['test_mean_tampered_dice'], new_mean_tampered_dice=new['mean_tampered_dice']))
if rows:
    check = pd.DataFrame(rows); display(check.round(4))
    worst = check['diff'].abs().max()
    print('PASS' if worst < 0.01 else 'CẢNH BÁO: lệch > 0.01, kiểm tra lại tiền xử lý', f'(max |diff| = {worst:.4f})')

## 6. Cross-dataset: Coverage (zero-shot — không fine-tune, không đổi ngưỡng)

Lần chạy đầu chỉ tạo manifest + **ảnh preview**. Kiểm tra: vùng đỏ phải nằm đúng vùng bị copy-move. Đúng thì đặt `COVERAGE_MASKS_CHECKED = True` ở cell CONFIG và chạy lại từ đây.

In [ ]:
COVERAGE_CSV = Path('/content/manifests/coverage_test.csv')
COVERAGE_PREVIEW = Path('/content/coverage_preview.png')

def find_coverage_root(base):
    for zip_path in base.rglob('*.zip'):          # repo có thể chứa file nén
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(zip_path.parent)
    for candidate in [base, *[p for p in base.rglob('*') if p.is_dir()]]:
        if (candidate / 'image').is_dir() and (candidate / 'mask').is_dir():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy thư mục chứa image/ và mask/ trong {base}')

if RUN_COVERAGE:
    coverage_base = Path('/content/coverage_repo')
    if not coverage_base.exists():
        run(['git', 'clone', '-q', 'https://github.com/wenbihan/coverage.git', coverage_base], cwd='/content')
    COVERAGE_ROOT = find_coverage_root(coverage_base)
    command = [sys.executable, 'scripts/build_external_manifest.py', 'coverage', '--dataset-root', COVERAGE_ROOT,
               '--output', COVERAGE_CSV, '--preview', COVERAGE_PREVIEW]
    if COVERAGE_MASKS_CHECKED:
        command.append('--masks-checked')
    run(command)
    from IPython.display import Image as IPImage, display
    display(IPImage(str(COVERAGE_PREVIEW)))

In [ ]:
if RUN_COVERAGE and COVERAGE_MASKS_CHECKED:
    evaluate(COVERAGE_CSV, 'coverage', ['--save-visuals', '6'])
elif RUN_COVERAGE:
    print('⚠ Chưa đánh giá Coverage: xem preview ở trên rồi đặt COVERAGE_MASKS_CHECKED = True.')

## 7. Bảng kết quả cho báo cáo

In [ ]:
run([sys.executable, 'scripts/summarize_results.py', '--eval-dir', EVAL_DIR, '--output-dir', TABLE_DIR])
from IPython.display import Markdown, display
display(Markdown('### Mean ± SD theo cấu hình\n' + (TABLE_DIR / 'mean_sd.md').read_text(encoding='utf-8')))
display(Markdown('### Generalization (train CASIA → test từng dataset)\n' + (TABLE_DIR / 'generalization.md').read_text(encoding='utf-8')))
paired = TABLE_DIR / 'paired_vs_rgb.csv'
if paired.exists():
    display(Markdown('### Chênh lệch so với RGB cùng seed (Noise/DCT − RGB)'))
    frame = pd.read_csv(paired)
    display(frame[['dataset', 'architecture', 'mode', 'seed', 'delta_pixel_micro_dice',
                   'delta_pixel_authentic_fp_pixel_pct', 'delta_image_f1', 'delta_image_auc']].round(4))

In [ ]:
# Confusion matrix (CASIA test) của từng lượt
rows = []
for summary_path in sorted((EVAL_DIR / 'casia_test').glob('*/summary.json')):
    s = json.loads(summary_path.read_text())
    i = s['image']
    rows.append(dict(run=s['run'], TN=i['tn'], FP=i['fp'], FN=i['fn'], TP=i['tp'],
                     accuracy=i['accuracy'], precision=i['precision'], recall=i['recall'], f1=i['f1'], auc=i['auc'],
                     threshold=s['thresholds']['image_threshold']))
display(pd.DataFrame(rows).round(4))

## 8. Ví dụ thành công / thất bại (cho phần phân tích lỗi)

In [ ]:
from IPython.display import Image as IPImage, display
# Model tốt nhất được chọn theo VALIDATION micro Dice (không theo test)
val_scores = []
for summary_path in (EVAL_DIR / 'casia_val').glob('*/summary.json'):
    s = json.loads(summary_path.read_text())
    val_scores.append((s['pixel']['micro_dice'], s['run'], s['checkpoint']))
val_scores.sort(reverse=True)
BEST_VAL_DICE, BEST_RUN, BEST_CHECKPOINT = val_scores[0]
print(f'Model tốt nhất theo validation: {BEST_RUN} (val micro Dice {BEST_VAL_DICE:.4f})')
print('Panel: ảnh | ground truth | bản đồ xác suất | dự đoán')
visuals = EVAL_DIR / 'casia_test' / BEST_RUN / 'visuals'
for group in ('best_tampered', 'worst_tampered', 'worst_authentic_fp'):
    files = sorted((visuals / group).glob('*.png'))[:3]
    print(f'\n== {group} ({len(files)}) ==')
    for f in files:
        print(f.name); display(IPImage(str(f), width=900))

## 9. Dự đoán một ảnh: Real/Fake, confidence, mask, bounding box, diện tích

Mặc định dùng một ảnh tampered trong CASIA test; đặt `UPLOAD_OWN_IMAGE = True` để upload ảnh của bạn.

In [ ]:
from PIL import Image
from forgery_localization.inference import Predictor, load_thresholds, render_overlay, report_text

thresholds_path = Path(BEST_CHECKPOINT).parent / 'thresholds.json'
predictor = Predictor(BEST_CHECKPOINT, thresholds=load_thresholds(thresholds_path))

image_paths = []
if UPLOAD_OWN_IMAGE:
    from google.colab import files
    image_paths = [Path(name) for name in files.upload()]
if not image_paths:
    image_paths = [next(r['image_path'] for r in test_ds.records if r['label'] == 'tampered')]

for path in image_paths:
    with Image.open(path) as opened:
        image = opened.convert('RGB')
    probabilities, mask, summary = predictor.predict(image)
    display(Markdown(f'**{Path(path).name}**\n\n' + report_text(summary, predictor.name)))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image); axes[0].set_title('Ảnh gốc')
    axes[1].imshow(probabilities, cmap='magma', vmin=0, vmax=1); axes[1].set_title('Bản đồ xác suất')
    axes[2].imshow(render_overlay(image, mask, summary)); axes[2].set_title('Mask + bounding box')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

## 10. Web demo (Gradio — link công khai)
Chọn model trong dropdown để so sánh RGB / Noise / DCT / SegFormer trên cùng một ảnh.

In [ ]:
if RUN_DEMO:
    import importlib.util
    spec = importlib.util.spec_from_file_location('demo_app', REPO / 'app' / 'demo.py')
    demo_app = importlib.util.module_from_spec(spec); spec.loader.exec_module(demo_app)
    seed0 = SEEDS[0]
    demo_checkpoints = [str(c) for c in (checkpoint_for(a, m, seed0) for a, m, s in PLAN if s == seed0) if c is not None]
    demo = demo_app.build_interface(demo_app.load_predictors(demo_checkpoints, 'auto'))
    demo.launch(share=True, debug=False)